# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）とhbox（頭部）** を使ってYOLOをファインチューニングします。1つのモデルで、`0: person`（vbox）と `1: head`（hbox）の2クラスを学習します。

GPU確認・Google Drive接続・2クラスのデータセット準備・事前学習モデル取得・学習と保存・再開まで実装しています。上から順に実行してください。セル6を実行すると学習が開始します。

学習処理の実装方針:

- 小規模学習は挟まず、設定したエポック数で学習します。
- 各エポック終了時に、再開に必要な状態を含むチェックポイントと学習結果をGoogle Driveへ保存します。
- セッション中断後は、最後に正常に保存できたエポックから再開します。エポック途中からの再開は対象にしません。
- 保存・再開処理はセル6に実装しています。

確認済みの保存場所:

- データ: `/content/drive/MyDrive/crowdhuman`
- 注釈: `raw/annotations/annotation_train.odgt`、`raw/annotations/annotation_val.odgt`（データフォルダからの相対パス）
- 保存済みデータの管理情報: `prepared/latest.json`（データフォルダからの相対パス）
- 管理情報の参照先: `prepared_9c497bb835b6435ea77e0992ab70b7d4.tar`（`prepared` 内）。設定は `vbox`。hboxを含むか、tarの中身は未確認です。

今後使用する保存先:

- 学習結果: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重み: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- ローカルデータ展開先: `/content/crowdhuman_vbox_yolo`


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    print("GPUが利用できません。ランタイムのタイプを確認してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. データセット準備の設定

Driveの `raw/archives` の4個のZIPと、`raw/annotations` のtrain・val注釈を使用します。
`prepared` のtarは管理情報がvbox設定のため、今回の2クラス変換には使用しません。
Driveの元データは変更しません。ローカルへの展開とラベル変換には時間・空き容量が必要です。

出力先は今回の準備処理専用にしてください。設定が一致する途中出力は再利用します。
Colabのローカルデータが失われた場合は、セル3〜4を再実行して準備し直します。


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/crowdhuman")
OUTPUT_DIR = Path("/content/crowdhuman_vbox_yolo")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("先にセル2でGoogle Driveを接続してください。")


## 4. 画像の展開と2クラスのYOLOラベル生成

元のtrain・val分割を維持し、次のルールで変換します。

- `tag=person` かつ `extra.ignore` が未指定または0の人物を対象にします。
- `0: person` はvbox、`1: head` はhboxです。頭部はさらに `head_attr.ignore` が未指定または0のものを使います。
- 座標 `[x, y, w, h]` を画像内にクリップし、YOLOの正規化座標に変換します。欠損・非有限値・幅高さが0以下・画像内の面積が0の枠は除きます。
- 空ラベルの画像も残します。除外数と各クラスの枠数は処理後に表示し、JSONにも保存します。

**無視枠をラベルから除いても、学習時にその領域の損失が無効になるわけではありません。** この出力は通常のYOLO用データで、CrowdHuman公式の無視領域対応や公式評価の実装ではありません。元注釈はDriveに保持します。
注釈形式の出典: [CrowdHuman公式仕様](https://www.crowdhuman.org/download.html)

ZIPは1個ずつローカルへコピーし、注釈に対応する画像だけ展開します。初回展開時はZIP読み取りのCRCチェックが働き、変換時は全画像をデコードします。同じ設定で再実行すると、サイズが一致する展開済み画像を再利用し、ラベルを再生成します。ZIP全体のハッシュ照合は行いません。
完了後、`images/{train,val}`、`labels/{train,val}`、`data.yaml`、`preparation_report.json` が出力されます。

**完了時の集計結果を共有してください。学習は実行しません。**


In [ ]:
from pathlib import Path
import json
import hashlib
import math
import shutil
import tempfile
import zipfile
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm


def read_records(path):
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            image_id = record.get("ID")
            if (not isinstance(image_id, str) or not image_id
                    or "/" in image_id or "\\" in image_id or image_id in (".", "..")):
                raise ValueError(f"不正な画像ID: {path}:{line_number}")
            if not isinstance(record.get("gtboxes"), list):
                raise ValueError(f"gtboxesがありません: {path}:{line_number}")
            yield record


def normalized_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x, y, w, h = map(float, box)
    except (TypeError, ValueError):
        return None
    if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:
        return None
    left, top = max(0.0, x), max(0.0, y)
    right, bottom = min(float(width), x + w), min(float(height), y + h)
    if right <= left or bottom <= top:
        return None
    return ((left + right) / (2 * width), (top + bottom) / (2 * height),
            (right - left) / width, (bottom - top) / height)


def labels_for_record(record, width, height):
    lines, counts = [], Counter()
    for obj in record["gtboxes"]:
        if obj.get("tag") != "person":
            counts["excluded_non_person"] += 1
            continue
        if (obj.get("extra") or {}).get("ignore", 0) != 0:
            counts["excluded_person_ignore"] += 1
            continue
        for class_id, key in ((0, "vbox"), (1, "hbox")):
            if class_id == 1 and (obj.get("head_attr") or {}).get("ignore", 0) != 0:
                counts["excluded_head_ignore"] += 1
                continue
            box = normalized_box(obj.get(key), width, height)
            if box is None:
                counts[f"invalid_{key}"] += 1
                continue
            lines.append(f"{class_id} " + " ".join(f"{value:.10f}" for value in box))
            counts[key] += 1
    return lines, counts


def atomic_text(path, text):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def prepare_dataset(data_dir, output_dir):
    annotation_paths = {
        split: data_dir / "raw" / "annotations" / f"annotation_{split}.odgt"
        for split in ("train", "val")
    }
    archive_paths = {
        "train": [data_dir / "raw" / "archives" / f"CrowdHuman_train{i:02d}.zip"
                  for i in (1, 2, 3)],
        "val": [data_dir / "raw" / "archives" / "CrowdHuman_val.zip"],
    }
    source_paths = list(annotation_paths.values()) + sum(archive_paths.values(), [])
    for path in source_paths:
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"必要なファイルがないか空です: {path}")

    config = {
        "schema": 1,
        "classes": {"0": "person_vbox", "1": "head_hbox"},
        "policy": "person_tag; extra.ignore=0; head_attr.ignore=0; clip; discard_invalid",
        "annotations": {split: sha256_file(path) for split, path in annotation_paths.items()},
        # ZIPはサイズと更新時刻を記録します。全体のハッシュ検証ではありません。
        "archives": {str(path): {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
                     for path in sum(archive_paths.values(), [])},
    }
    marker = output_dir / ".build_config.json"
    if output_dir.exists() and any(output_dir.iterdir()):
        if not marker.is_file() or json.loads(marker.read_text()) != config:
            raise RuntimeError("出力先に別設定または管理対象外のデータがあります。OUTPUT_DIRを別の空フォルダに変更してください。")
    output_dir.mkdir(parents=True, exist_ok=True)
    atomic_text(marker, json.dumps(config, ensure_ascii=False, indent=2))

    ids = {}
    for split, path in annotation_paths.items():
        ids[split] = set()
        for record in read_records(path):
            if record["ID"] in ids[split]:
                raise ValueError(f"注釈の画像ID重複: {split}/{record['ID']}")
            ids[split].add(record["ID"])
        if not ids[split]:
            raise ValueError(f"注釈が空です: {path}")
        for kind in ("images", "labels"):
            (output_dir / kind / split).mkdir(parents=True, exist_ok=True)
    if ids["train"] & ids["val"]:
        raise ValueError("trainとvalで画像IDが重複しています。")

    # ZIPの目録を照合します。全画像の展開前に欠落・重複を検出します。
    plans, required_bytes, largest_zip = [], 0, 0
    for split, paths in archive_paths.items():
        found = set()
        for path in paths:
            entries = []
            with zipfile.ZipFile(path) as archive:
                for member in archive.infolist():
                    name = Path(member.filename)
                    if member.is_dir() or name.suffix.lower() != ".jpg" or name.stem not in ids[split]:
                        continue
                    if name.stem in found:
                        raise ValueError(f"ZIP内の画像ID重複: {split}/{name.stem}")
                    found.add(name.stem)
                    target = output_dir / "images" / split / f"{name.stem}.jpg"
                    if not target.is_file() or target.stat().st_size != member.file_size:
                        required_bytes += member.file_size
                    entries.append((member.filename, target, member.file_size))
            plans.append((path, entries))
            largest_zip = max(largest_zip, path.stat().st_size)
        missing = ids[split] - found
        if missing:
            raise ValueError(f"{split}: ZIPに画像が{len(missing)}件不足。例: {sorted(missing)[:5]}")
        print(f"{split}: 注釈とZIPの画像ID一致 {len(found):,}件")

    free = shutil.disk_usage(output_dir).free
    needed = required_bytes + largest_zip + 2 * 1024**3
    print(f"ローカル空き: {free / 1024**3:.2f} GiB / 必要目安: {needed / 1024**3:.2f} GiB")
    if free < needed:
        raise RuntimeError("ローカル空き容量が不足しています。Driveの元データは変更していません。")

    # 一度に1個だけZIPをローカルへコピーし、必要な画像だけ取り出します。
    # 一時領域のみ自動削除。出力画像は一時ファイル経由で確定します。
    with tempfile.TemporaryDirectory(prefix="crowdhuman_zip_", dir=output_dir.parent) as cache:
        for source, entries in plans:
            pending = [(name, target, size) for name, target, size in entries
                       if not target.is_file() or target.stat().st_size != size]
            if not pending:
                print(f"{source.name}: 展開済みのためスキップ")
                continue
            local_zip = Path(cache) / source.name
            print(f"ローカルへコピー中: {source.name}")
            shutil.copyfile(source, local_zip)
            with zipfile.ZipFile(local_zip) as archive:
                for name, target, size in tqdm(pending, desc=source.name):
                    temporary = target.with_name(target.name + ".part")
                    with archive.open(name) as src, temporary.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                    if temporary.stat().st_size != size:
                        raise IOError(f"画像サイズ不一致: {name}")
                    temporary.replace(target)
            local_zip.unlink()

    summary = {}
    for split, annotation_path in annotation_paths.items():
        counts = Counter()
        for record in tqdm(read_records(annotation_path), total=len(ids[split]), desc=f"{split} ラベル変換"):
            image_path = output_dir / "images" / split / f"{record['ID']}.jpg"
            # 全画像をデコードし、破損や読み取り不能を検出します。
            with Image.open(image_path) as image:
                image.load()
                width, height = image.size
            lines, box_counts = labels_for_record(record, width, height)
            counts.update(box_counts)
            counts["images"] += 1
            counts["empty_labels"] += int(not lines)
            label_path = output_dir / "labels" / split / f"{record['ID']}.txt"
            atomic_text(label_path, "\n".join(lines) + ("\n" if lines else ""))
        summary[split] = dict(counts)
        if counts["vbox"] == 0 or counts["hbox"] == 0:
            raise ValueError(f"{split}: personまたはheadの有効ラベルが0件です。")
        if len(list((output_dir / "images" / split).glob("*.jpg"))) != len(ids[split]):
            raise ValueError(f"{split}: 出力先に余分な画像があります。別の空フォルダを指定してください。")
        if len(list((output_dir / "labels" / split).glob("*.txt"))) != len(ids[split]):
            raise ValueError(f"{split}: 出力ラベル数が一致しません。")

    yaml_path = output_dir / "data.yaml"
    yaml_text = (f"path: {json.dumps(str(output_dir.resolve()))}\n"
                 "train: images/train\nval: images/val\nnames:\n  0: person\n  1: head\n")
    atomic_text(yaml_path, yaml_text)
    atomic_text(output_dir / "preparation_report.json", json.dumps(summary, ensure_ascii=False, indent=2))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"準備完了。学習用YAML: {yaml_path}")
    return yaml_path

DATA_YAML = prepare_dataset(DATA_DIR, OUTPUT_DIR)


## 5. Ultralyticsの導入と事前学習モデルの取得

元ノートブックと同じ、COCO事前学習済みの `yolo26n.pt` を使用します。
再実行・再開時に同じUltralyticsを使うため、バージョンを `8.4.0` に固定します。
初回はインターネットから取得して読み込みを確認した後、Driveの `crowdhuman_vbox_ft/weights` に保存します。保存済みの場合は再利用します。
重みのSHA-256とPyTorch・CUDA・Ultralyticsのバージョンを `pretrained_environment.json` に記録します。SHA-256はコピーの整合性確認と記録のためのもので、配布元の署名検証ではありません。

この段階ではモデルの検出ヘッドはCOCO用です。person・headの2クラスへの変更は、セル6の学習処理が `data.yaml` に基づいて行います。学習・推論は実行しません。

依存パッケージの変更に伴って再起動を求められた場合は、ランタイムを再起動して上から実行してください。Colabローカルの画像が残っていれば、セル4で再利用できます。

参照: [YOLO26公式ドキュメント](https://docs.ultralytics.com/models/yolo26/)、[固定バージョンの配布情報](https://pypi.org/project/ultralytics/8.4.0/)

モデルを取得できたらセル6へ進み、学習条件を設定してください。


In [ ]:
import importlib.metadata
import json
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ULTRALYTICS_VERSION = "8.4.0"
MODEL_NAME = "yolo26n.pt"
WEIGHTS_DIR = Path("/content/drive/MyDrive/crowdhuman_vbox_ft/weights")
PRETRAINED_PATH = WEIGHTS_DIR / MODEL_NAME

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("先にセル2でGoogle Driveを接続してください。")
if not Path(DATA_YAML).is_file():
    raise RuntimeError("先にセル3〜4でデータセットを準備してください。")

loaded = sys.modules.get("ultralytics")
if loaded is not None and loaded.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError("別バージョンのUltralyticsが読み込まれています。ランタイムを再起動してから実行してください。")
try:
    installed_version = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    installed_version = None
if installed_version != ULTRALYTICS_VERSION:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", f"ultralytics=={ULTRALYTICS_VERSION}"],
        check=True,
    )

import torch
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.downloads import safe_download

if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError("バージョンが反映されていません。ランタイムを再起動してください。")

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
if PRETRAINED_PATH.exists():
    if not PRETRAINED_PATH.is_file() or PRETRAINED_PATH.stat().st_size == 0:
        raise RuntimeError(f"保存済み重みが不正です。ファイルを確認してください: {PRETRAINED_PATH}")
    print(f"Driveの保存済み重みを使用: {PRETRAINED_PATH}")
    model = YOLO(str(PRETRAINED_PATH), task="detect")
else:
    # ローカルで取得・読み込みを確認してからDriveへコピーします。
    url = f"https://github.com/ultralytics/assets/releases/download/v8.4.0/{MODEL_NAME}"
    with tempfile.TemporaryDirectory(prefix="crowdhuman_model_") as temporary:
        local_path = Path(temporary) / MODEL_NAME
        safe_download(url=url, file=str(local_path), unzip=False, min_bytes=100_000)
        if not local_path.is_file() or local_path.stat().st_size < 100_000:
            raise RuntimeError("公式配布先から重みを取得できませんでした。時間を置いて再実行してください。")
        model = YOLO(str(local_path), task="detect")
        partial_path = PRETRAINED_PATH.with_suffix(".pt.part")
        shutil.copyfile(local_path, partial_path)
        if sha256_file(local_path) != sha256_file(partial_path):
            raise IOError("Driveへのコピー後に重みのハッシュが一致しません。再実行してください。")
        partial_path.replace(PRETRAINED_PATH)
    # 一時パスをモデルに残さないよう、Driveの確定ファイルを読み直します。
    model = YOLO(str(PRETRAINED_PATH), task="detect")

if model.task != "detect":
    raise ValueError("物体検出モデルではありません。")
environment = {
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "model": MODEL_NAME,
    "weights_sha256": sha256_file(PRETRAINED_PATH),
}
atomic_text(WEIGHTS_DIR / "pretrained_environment.json", json.dumps(environment, ensure_ascii=False, indent=2))
print(json.dumps(environment, ensure_ascii=False, indent=2))
print(f"事前学習重み: {PRETRAINED_PATH}")
print(f"学習用データ: {DATA_YAML}")
model.info()
print("モデル取得完了。現在の検出ヘッドはCOCO用です。2クラスへの変更は学習開始時に行います。")


## 6. 学習・エポックごとのDrive保存・再開

**このコードセルを実行すると学習が開始します。** 初回実行前に冒頭の設定を変更してください。
初期値は `EPOCHS=100`、`IMGSZ=640`、`BATCH=8`、`WORKERS=2` です。
1モデル・2クラスを全trainデータで学習し、毎エポックvalで評価します。小規模学習や段階的な規模変更は行いません。
AdamW（初期学習率0.001）を使用し、早期終了は無効にします。これらも学習関数内の `train_args` で変更できます。
GPUメモリ不足の場合は、初回保存前ならBATCHを下げて新しいRUN_NAMEで開始してください。

保存先は `/content/drive/MyDrive/crowdhuman_vbox_ft/runs/<RUN_NAME>` です。

- `epochs/epoch_000001/` など: 各エポックの再開用重み（オプティマイザー・EMA・AMP scalerを含む）、その時点のbest重み、累積 `results.csv`、学習引数、ハッシュ付き管理情報。
- `latest.json`: 最後にDriveへのコピーとハッシュ確認が完了したエポック。保存途中では更新しません。
- `run_config.json`: データセット・事前学習重み・学習設定の照合用情報。
- `final/`: 正常終了時の推論用best/last重み、CSV・図など。再開用の各エポック保存とは分けて保持します。

同じRUN_NAME・設定でセルを再実行すると、latest.jsonが指すチェックポイントから**次のエポック**を開始します。
`EPOCHS` は追加回数ではなく総エポック数です。エポック途中の処理は保存されず、標準の再開処理は中断なしの実行との完全な数値一致を保証しません。
セッションが失われた場合はセル1〜5でGPU・Drive・データ・ライブラリを復元してから、このセルを実行してください。
保存済みの総エポック数に達している場合は学習を再開しません。最終評価やfinalのコピー前に中断されても、最終エポックの保存済み重みは利用できます。

学習設定またはデータを変える場合は新しいRUN_NAMEを指定してください。既存の実験を上書き・自動削除しません。
各エポックを残すため、Driveの使用容量はエポック数に応じて増えます。Driveへの保存失敗時は学習を停止し、直前の正常保存分を再開先として残します。
再開はUltralytics 8.4.0の標準処理を使用します。評価は通常のYOLO指標で、CrowdHuman公式の無視領域対応評価ではありません。

参照: [学習・再開](https://docs.ultralytics.com/modes/train/)、[保存コールバック](https://docs.ultralytics.com/usage/callbacks/)


In [ ]:
# 初回学習前に変更してください。同じRUN_NAMEでの再開時は設定を維持します。
EPOCHS = 100
IMGSZ = 640
BATCH = 8
WORKERS = 2
RUN_NAME = "yolo26n_person_head"

import hashlib
import json
import shutil
from pathlib import Path

import torch
import ultralytics
import yaml
from ultralytics import YOLO

DRIVE_RUNS_DIR = Path("/content/drive/MyDrive/crowdhuman_vbox_ft/runs")
LOCAL_RUNS_DIR = Path("/content/crowdhuman_training")


def copy_verified(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    partial = target.with_name(target.name + ".part")
    shutil.copyfile(source, partial)
    digest = sha256_file(source)
    if sha256_file(partial) != digest:
        raise IOError(f"コピーのハッシュ不一致: {target}")
    partial.replace(target)
    return digest


def run_training(data_yaml, pretrained_path, drive_runs, local_runs,
                 run_name, epochs, imgsz, batch, workers, *, device=0, amp=True,
                 plots=True, require_cuda=True):
    if ultralytics.__version__ != "8.4.0":
        raise RuntimeError("セル5でUltralytics 8.4.0を導入してください。")
    if require_cuda and not torch.cuda.is_available():
        raise RuntimeError("GPUが利用できません。Colabのランタイム設定を確認してください。")
    if type(epochs) is not int or epochs < 1:
        raise ValueError("EPOCHSは1以上の整数にしてください。")
    if type(batch) is not int or batch < 1:
        raise ValueError("BATCHは1以上の整数にしてください。")
    if not run_name or Path(run_name).name != run_name or run_name in (".", "..") or "\\" in run_name:
        raise ValueError("RUN_NAMEには単一のフォルダ名を指定してください。")
    data_yaml, pretrained_path = Path(data_yaml), Path(pretrained_path)
    dataset_dir = data_yaml.parent
    dataset_config = dataset_dir / ".build_config.json"
    dataset_report = dataset_dir / "preparation_report.json"
    for path in (data_yaml, pretrained_path, dataset_config, dataset_report):
        if not path.is_file():
            raise FileNotFoundError(f"先にセル3〜5を実行してください: {path}")
    names = yaml.safe_load(data_yaml.read_text())["names"]
    if names != {0: "person", 1: "head"}:
        raise ValueError("data.yamlのクラスがperson・headの2クラスではありません。")

    drive_run = Path(drive_runs) / run_name
    local_run = Path(local_runs).resolve() / run_name
    train_args = {
        "data": str(data_yaml.resolve()), "epochs": epochs, "imgsz": imgsz,
        "batch": batch, "workers": workers, "device": device,
        "project": str(local_run.parent), "name": run_name, "exist_ok": True,
        "optimizer": "AdamW", "lr0": 0.001, "seed": 0,
        "patience": 0, "save": True, "save_period": -1,
        "val": True, "plots": plots, "amp": amp, "cache": False,
        "close_mosaic": min(10, epochs),
    }
    config = {
        "schema": 1, "ultralytics": ultralytics.__version__,
        "pretrained_sha256": sha256_file(pretrained_path),
        "dataset_config_sha256": sha256_file(dataset_config),
        "data_yaml_sha256": sha256_file(data_yaml), "train_args": train_args,
    }
    config_text = json.dumps(config, sort_keys=True, ensure_ascii=False)
    config_hash = hashlib.sha256(config_text.encode()).hexdigest()
    for root, marker_name in ((drive_run, "run_config.json"), (local_run, ".run_config.json")):
        marker = root / marker_name
        if root.exists() and any(root.iterdir()):
            if not marker.is_file() or json.loads(marker.read_text()) != config:
                raise RuntimeError(f"別設定または管理対象外の保存先です: {root}。新しいRUN_NAMEを指定してください。")
        root.mkdir(parents=True, exist_ok=True)
        atomic_text(marker, json.dumps(config, ensure_ascii=False, indent=2))

    latest_path = drive_run / "latest.json"
    resume_path = None
    if latest_path.is_file():
        latest = json.loads(latest_path.read_text())
        completed = latest["completed_epochs"]
        if latest.get("config_sha256") != config_hash or type(completed) is not int or not 1 <= completed <= epochs:
            raise ValueError("再開用の管理情報が学習設定と一致しません。")
        snapshot = drive_run / "epochs" / f"epoch_{completed:06d}"
        metadata = json.loads((snapshot / "snapshot.json").read_text())
        if metadata.get("config_sha256") != config_hash or metadata.get("completed_epochs") != completed:
            raise ValueError("保存済みエポックの管理情報が一致しません。")
        expected_files = {"checkpoint.pt", "best.pt", "results.csv", "args.yaml"}
        if set(metadata["files"]) != expected_files:
            raise ValueError("保存済みエポックの構成が不正です。")
        for name, digest in metadata["files"].items():
            path = snapshot / name
            if not path.is_file() or sha256_file(path) != digest:
                raise IOError(f"保存済みファイルの検証に失敗しました: {path}")
        if completed == epochs:
            print(f"指定した{epochs}エポックは学習・保存済みです: {snapshot}")
            print("再学習は行いません。最終成果物が未作成の場合も、このチェックポイントを利用できます。")
            return snapshot / "checkpoint.pt"
        resume_path = local_run / "weights" / "last.pt"
        copy_verified(snapshot / "checkpoint.pt", resume_path)
        copy_verified(snapshot / "best.pt", local_run / "weights" / "best.pt")
        copy_verified(snapshot / "results.csv", local_run / "results.csv")
        copy_verified(snapshot / "args.yaml", local_run / "args.yaml")
        checkpoint = torch.load(resume_path, map_location="cpu", weights_only=False)
        if checkpoint.get("epoch") != completed - 1 or checkpoint.get("optimizer") is None:
            raise ValueError("チェックポイントに再開用の状態がありません。")
        del checkpoint
        print(f"{completed}エポック保存済み。{completed + 1}エポック目から再開します。")
    else:
        # 初回エポックの保存前に中断した場合、未確定のローカルCSVを作り直します。
        uncommitted_csv = local_run / "results.csv"
        if uncommitted_csv.exists():
            uncommitted_csv.replace(local_run / "results.csv.uncommitted")
        print(f"新規学習: {epochs}エポック / 保存先: {drive_run}")

    def save_epoch(trainer):
        completed = trainer.epoch + 1
        snapshot = drive_run / "epochs" / f"epoch_{completed:06d}"
        files = {
            "checkpoint.pt": Path(trainer.last), "best.pt": Path(trainer.best),
            "results.csv": Path(trainer.csv), "args.yaml": Path(trainer.save_dir) / "args.yaml",
        }
        digests = {name: copy_verified(path, snapshot / name) for name, path in files.items()}
        metadata = {
            "completed_epochs": completed, "config_sha256": config_hash,
            "files": digests, "torch": str(torch.__version__), "cuda": torch.version.cuda,
        }
        atomic_text(snapshot / "snapshot.json", json.dumps(metadata, ensure_ascii=False, indent=2))
        # 全ファイルのコピーとハッシュ確認が終わるまで再開先を更新しません。
        atomic_text(latest_path, json.dumps({
            "completed_epochs": completed, "config_sha256": config_hash,
        }, indent=2))
        print(f"\nDrive保存完了: epoch {completed}/{epochs} → {snapshot}")

    def save_final(trainer):
        # Ultralyticsの最終処理で軽量化された重み・図を、再開用とは別に保存します。
        final_dir = drive_run / "final"
        for path in Path(trainer.save_dir).iterdir():
            if path.is_file() and path.suffix in (".csv", ".yaml", ".png", ".jpg", ".json"):
                copy_verified(path, final_dir / path.name)
        for path in (Path(trainer.last), Path(trainer.best)):
            copy_verified(path, final_dir / "weights" / path.name)
        atomic_text(final_dir / "completed.json", json.dumps({
            "completed_epochs": trainer.epoch + 1, "config_sha256": config_hash,
        }, indent=2))
        print(f"最終成果物の保存完了: {final_dir}")

    # 毎回新しいインスタンスにして、セル再実行時のコールバック重複を避けます。
    training_model = YOLO(str(resume_path or pretrained_path), task="detect")
    training_model.add_callback("on_model_save", save_epoch)
    training_model.add_callback("on_train_end", save_final)
    if resume_path is not None:
        # 固定バージョンでは再開時の総エポック数・学習率などをチェックポイントから復元します。
        training_model.train(resume=str(resume_path), **train_args)
    else:
        training_model.train(**train_args)
    return drive_run / "final" / "weights" / "best.pt"


TRAINED_WEIGHTS = run_training(
    DATA_YAML, PRETRAINED_PATH, DRIVE_RUNS_DIR, LOCAL_RUNS_DIR,
    RUN_NAME, EPOCHS, IMGSZ, BATCH, WORKERS,
)
